In [ ]:
import pandas as pd
from pathlib import Path
from reed import utils
import tarfile

In [ ]:
config = utils.Config()

In [ ]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"

## Define formulas

In [ ]:
wwr_pmg11007 = monitoring_data_path/'WWR - PMG-11007'

In [98]:
files_with_energy_data = {}

for tar_path in wwr_pmg11007.glob("*.tar.gz"):
    try:
        with tarfile.open(tar_path) as tar:
            member = tar.getmembers()[0]  # assume one CSV inside
            # read only first few rows to inspect columns
            df_preview = pd.read_csv(tar.extractfile(member), nrows=5)
            files_with_energy_data[tar_path.name] = "Power_Active" in df_preview.columns
    except Exception as e:
        files_with_energy_data[tar_path.name] = f"Error: {e}"

files_with_energy_data

{'OF10001_G_TWB_7SZ421_7DTZHE.tar.gz': False,
 'OF10001_G_TWB_7SZ421_92SQBY.tar.gz': True,
 'OF10001_G_TWB_7SZ421_BXCZ3M.tar.gz': False,
 'OF10001_G_TWB_7SZ421_TYZBPS.tar.gz': False,
 'OF10002_G_TWB_7SZ421_7DTZHE.tar.gz': False,
 'OF10002_G_TWB_7SZ421_92SQBY.tar.gz': True,
 'OF10002_G_TWB_7SZ421_BXCZ3M.tar.gz': False,
 'OF10002_G_TWB_7SZ421_TYZBPS.tar.gz': False}

In [ ]:
def load_wwr(file_number):
    paths = list(wwr_pmg11007.glob("*.gz"))
    wwr_path = paths[file_number]

    wwr_df = pd.read_csv(wwr_path)
    wwr_df["timestamp"] = pd.to_datetime(wwr_df["timestamp"])

    return wwr_df

In [82]:
def analyse_wwr(wwr_df, keep_columns):
    df = wwr_df.copy()
    df = df[keep_columns].copy()

    # Ensure timestamps are datetimes
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Sort and compute time delta between samples
    df = df.sort_values("timestamp")
    df["dt_seconds"] = df["timestamp"].diff().dt.total_seconds().fillna(0)

    # Correct energy calculation:
    # Energy (kWh) = Power(W) × time(s) / 3,600,000
    df["energy_consumption_kWh"] = df["Power_Active"] * df["dt_seconds"] / 3600000

    # Aggregate by tool
    df_bytool = (
        df.groupby("Tool_Number", as_index=False)
            .agg({
                "energy_consumption_kWh": "sum",
                "Power_Active": "mean",
                "dt_seconds":"sum"
            })
        )

    return df_bytool

In [ ]:
keep_columns = ["Tool_Number", "Power_Active", "timestamp"]

## First Operation: OF10001

In [88]:
# operation OF10001 - WWR
file001 = load_wwr(0)
file002 = load_wwr(1) # has energy data
file003 = load_wwr(2)
file004 = load_wwr(3) # has tool data

In [89]:
file004.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276782 entries, 0 to 276781
Data columns (total 28 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   Feed_Override            276782 non-null  float64            
 1   Feed_Rate_Actual         276782 non-null  float64            
 2   Feed_Rate_Commanded      276782 non-null  float64            
 3   Head_Angular_On          0 non-null       float64            
 4   Head_Auto_On             0 non-null       float64            
 5   Head_Boring_On           0 non-null       float64            
 6   Offset_X                 0 non-null       float64            
 7   Offset_Y                 0 non-null       float64            
 8   Offset_Z                 0 non-null       float64            
 9   Operation_Mode           276782 non-null  float64            
 10  Position_MCS_A           0 non-null       float64            
 11  Position_MCS_

In [90]:
file004.head()

,Feed_Override,Feed_Rate_Actual,Feed_Rate_Commanded,Head_Angular_On,Head_Auto_On,Head_Boring_On,Offset_X,Offset_Y,Offset_Z,Operation_Mode,...,Spindle_Speed_Commanded,Spindle_Speed_Override,Temperature_Head,Temperature_Room,Temperature_Y,Temperature_Z,Tool_Length,Tool_Number,Tool_Radius,timestamp
0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,16.0,...,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN,2025-02-11 08:34:36.696000+00:00
1,5.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,16.0,...,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN,2025-02-11 08:34:37.696000+00:00
2,31.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,16.0,...,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN,2025-02-11 08:34:38.696000+00:00
3,33.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,16.0,...,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN,2025-02-11 08:34:39.696000+00:00
4,33.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,16.0,...,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN,2025-02-11 08:34:40.696000+00:00


In [ ]:
# sort the relevant files by timestamp
file002 = file002.sort_values("timestamp")
file004 = file004.sort_values("timestamp")

In [84]:
# merge df for file002 and file004
WWR_OF10001_df = pd.merge_asof(file002, file004, on="timestamp", direction="nearest", tolerance=pd.Timedelta("300ms"))
WWR_OF10001_df.head(20)

,Energy_Total,Enery_Hours,Power_Active,Power_Apparent,Power_Factor,Power_Reactive,timestamp,Feed_Override,Feed_Rate_Actual,Feed_Rate_Commanded,...,Spindle_Speed_Actual,Spindle_Speed_Commanded,Spindle_Speed_Override,Temperature_Head,Temperature_Room,Temperature_Y,Temperature_Z,Tool_Length,Tool_Number,Tool_Radius
0,192141848.0,2658105.0,5950.0293,6082.1510,-0.978277,-1260.8372,2025-02-11 08:34:37.609000+00:00,5.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
1,192141851.0,2658105.0,7726.6523,7842.5130,-0.985227,-1343.0791,2025-02-11 08:34:38.609000+00:00,31.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
2,192141853.0,2658105.0,8022.6973,8861.2400,-0.905370,-3762.6992,2025-02-11 08:34:39.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
3,192141855.0,2658105.0,8137.1570,8255.3630,-0.985681,-1392.0126,2025-02-11 08:34:40.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
4,192141857.0,2658105.0,6060.3096,6207.0810,-0.976354,-1341.8284,2025-02-11 08:34:41.609000+00:00,28.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
5,192141859.0,2658105.0,6135.1177,6264.9634,-0.979274,-1268.8951,2025-02-11 08:34:42.609000+00:00,15.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
6,192141860.0,2658105.0,5806.3830,5943.1580,-0.976986,-1267.6952,2025-02-11 08:34:43.609000+00:00,41.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
7,192141862.0,2658105.0,5967.4550,6119.6380,-0.975132,-1356.2646,2025-02-11 08:34:44.609000+00:00,44.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
8,192141864.0,2658105.0,5744.8790,5911.6772,-0.971785,-1394.3799,2025-02-11 08:34:45.609000+00:00,41.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
9,192141865.0,2658105.0,5695.3574,6729.9680,-0.846268,-3585.4380,2025-02-11 08:34:46.609000+00:00,14.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN


In [93]:
WWR_OF10001_df.sort_values("timestamp")

,Energy_Total,Enery_Hours,Power_Active,Power_Apparent,Power_Factor,Power_Reactive,timestamp,Feed_Override,Feed_Rate_Actual,Feed_Rate_Commanded,...,Spindle_Speed_Actual,Spindle_Speed_Commanded,Spindle_Speed_Override,Temperature_Head,Temperature_Room,Temperature_Y,Temperature_Z,Tool_Length,Tool_Number,Tool_Radius
0,192141848.0,2658105.0,5950.0293,6082.1510,-0.978277,-1260.8372,2025-02-11 08:34:37.609000+00:00,5.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
1,192141851.0,2658105.0,7726.6523,7842.5130,-0.985227,-1343.0791,2025-02-11 08:34:38.609000+00:00,31.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
2,192141853.0,2658105.0,8022.6973,8861.2400,-0.905370,-3762.6992,2025-02-11 08:34:39.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
3,192141855.0,2658105.0,8137.1570,8255.3630,-0.985681,-1392.0126,2025-02-11 08:34:40.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
4,192141857.0,2658105.0,6060.3096,6207.0810,-0.976354,-1341.8284,2025-02-11 08:34:41.609000+00:00,28.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284994,192951307.0,2666017.0,9583.7580,10441.3850,0.917863,4144.1646,2025-02-14 15:47:19.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN
284995,192951310.0,2666017.0,9648.0020,10485.8570,0.920097,4107.2207,2025-02-14 15:47:20.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN
284996,192951311.0,2666017.0,6039.5760,6124.7900,-0.986087,-1018.1235,2025-02-14 15:47:21.705000+00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
284997,192951313.0,2666017.0,5924.6484,6029.8050,-0.982560,-1121.2013,2025-02-14 15:47:22.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN


In [94]:
WWR_OF10001_df.head()

,Energy_Total,Enery_Hours,Power_Active,Power_Apparent,Power_Factor,Power_Reactive,timestamp,Feed_Override,Feed_Rate_Actual,Feed_Rate_Commanded,...,Spindle_Speed_Actual,Spindle_Speed_Commanded,Spindle_Speed_Override,Temperature_Head,Temperature_Room,Temperature_Y,Temperature_Z,Tool_Length,Tool_Number,Tool_Radius
0,192141848.0,2658105.0,5950.0293,6082.151,-0.978277,-1260.8372,2025-02-11 08:34:37.609000+00:00,5.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
1,192141851.0,2658105.0,7726.6523,7842.513,-0.985227,-1343.0791,2025-02-11 08:34:38.609000+00:00,31.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
2,192141853.0,2658105.0,8022.6973,8861.240,-0.905370,-3762.6992,2025-02-11 08:34:39.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
3,192141855.0,2658105.0,8137.1570,8255.363,-0.985681,-1392.0126,2025-02-11 08:34:40.609000+00:00,33.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN
4,192141857.0,2658105.0,6060.3096,6207.081,-0.976354,-1341.8284,2025-02-11 08:34:41.609000+00:00,28.0,0.0,0.0,...,0.0,NaN,100.0,233.0,0.0,205.0,237.0,NaN,0.0,NaN


In [96]:
WWR_OF10001_df.tail()

,Energy_Total,Enery_Hours,Power_Active,Power_Apparent,Power_Factor,Power_Reactive,timestamp,Feed_Override,Feed_Rate_Actual,Feed_Rate_Commanded,...,Spindle_Speed_Actual,Spindle_Speed_Commanded,Spindle_Speed_Override,Temperature_Head,Temperature_Room,Temperature_Y,Temperature_Z,Tool_Length,Tool_Number,Tool_Radius
284994,192951307.0,2666017.0,9583.7580,10441.3850,0.917863,4144.1646,2025-02-14 15:47:19.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN
284995,192951310.0,2666017.0,9648.0020,10485.8570,0.920097,4107.2207,2025-02-14 15:47:20.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN
284996,192951311.0,2666017.0,6039.5760,6124.7900,-0.986087,-1018.1235,2025-02-14 15:47:21.705000+00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
284997,192951313.0,2666017.0,5924.6484,6029.8050,-0.982560,-1121.2013,2025-02-14 15:47:22.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN
284998,192951315.0,2666017.0,5914.7140,6023.9966,-0.981859,-1142.2327,2025-02-14 15:47:23.705000+00:00,100.0,0.0,150.0,...,199.0,NaN,100.0,192.0,0.0,194.0,216.0,NaN,2532.0,NaN


In [91]:
192951315.0-192141848.0

809467.0

In [87]:
# results for operation OF10002
WWR_OF10001_df_bytool = analyse_wwr(WWR_OF10001_df, keep_columns)
WWR_OF10001_df_bytool

,Tool_Number,energy_consumption_kWh,Power_Active,dt_seconds
0,0.0,30.112371,2907.553672,37288.007
1,81.0,3.450547,12688.411042,979.000
2,82.0,4.866510,10585.775663,1655.000
3,100.0,16.254928,7556.880686,7743.999
4,102.0,16.848635,9885.395542,6136.000
5,2424.0,0.575642,10519.337647,197.000
6,2427.0,12.325427,21890.230391,2027.000
7,2429.0,0.306580,13625.780105,81.000
8,2432.0,2.699714,9435.893309,1030.000
9,2433.0,9.035456,11803.166575,2756.000


In [92]:
WWR_OF10001_df_bytool["energy_consumption_kWh"].sum()

786.6316812851458

## Sceond Operation: OF10002

In [ ]:
# operation OF10002 - WWR
file005 = load_wwr(4)
file006 = load_wwr(5) # has energy data
file007 = load_wwr(6)
file008 = load_wwr(7) # has tool data

In [ ]:
file006.info()

In [ ]:
# sort the relevant files by timestamp
file006 = file006.sort_values("timestamp")
file008 = file008.sort_values("timestamp")

In [ ]:
# merge df for file002 and file004
WWR_OF10002_df = pd.merge_asof(file006, file008, on="timestamp", direction="nearest", tolerance=pd.Timedelta("300ms"))
WWR_OF10002_df.head(20)

In [ ]:
WWR_OF10002_df.sort_values("timestamp")

In [ ]:
192955194.0-192951323.0	

In [76]:
# results for operation OF10002
WWR_OF10002_df_bytool = analyse_wwr(WWR_OF10002_df, keep_columns)
WWR_OF10002_df_bytool

,Tool_Number,energy_consumption_kWh,Power_Active
0,0.0,0.761267,8897.931772
1,2455.0,1.768760,10021.076474
2,2695.0,1.215456,10271.477921


In [77]:
WWR_OF10002_df_bytool["energy_consumption_kWh"].sum()

3.7454841082732226

In [ ]:
# tools change over time
import matplotlib.pyplot as plt

def plot_energy_vs_time(*dfs, label_prefix="WWR_OF10002_df"):
    plt.figure(figsize=(14,6))

    for i, df in enumerate(dfs, start=1):
        # calculate elapsed time since start of dataset
        df = df.copy()
        df["energy_consumption"] = df["Power_Active"] / 3600
        df["elapsed_seconds"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
        
        # plot meaningfully, fx energy_consumption (or Tool_Number)
        plt.plot(df["elapsed_seconds"], df["energy_consumption"], label=f"{label_prefix}{i}")

    plt.title("Energy consumption over elapsed time")
    plt.xlabel("Time since start (seconds)")
    plt.ylabel("Energy consumption (kWh per second)")
    plt.legend()
    plt.show()


# --- USE IT ---
plot_energy_vs_time(WWR_OF10002_df)